# Load raw datasets

In [118]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [119]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.eda_utils import (
    clean_headers,
    load_gh_aw_config_files,
    create_gh_aw_id,
    parse_colected_date,
    parse_md_SHA,
    parse_md_url,
    parse_repo,
    parse_workflow,
    parse_yml_SHA,
    parse_yml_filename,
    parse_yml_url,
)

In [142]:
df = load_gh_aw_config_files("data/gh-aw_raw")
df.shape

(5501, 2)

In [143]:
df.head()

,workflow_markdown,workflow_yml
0,# Repo: 0GiS0/github-agentic-workflows-demos\n...,# Repo: 0GiS0/github-agentic-workflows-demos\n...
1,# Repo: 0GiS0/jsconf-2026\n# Workflow: issue-q...,# Repo: 0GiS0/jsconf-2026\n# Workflow: issue-q...
2,# Repo: AIS-Commercial-Business-Unit/RiskInsur...,# Repo: AIS-Commercial-Business-Unit/RiskInsur...
3,# Repo: AdaCore/z3\n# Workflow: daily-backlog-...,# Repo: AdaCore/z3\n# Workflow: daily-backlog-...
4,# Repo: AdaCore/z3\n# Workflow: pr-fix\n# MD U...,# Repo: AdaCore/z3\n# Workflow: pr-fix\n# MD U...


# Parse fields and remove metadata in the header

In [ ]:
df["repo"] = df["workflow_markdown"].apply(parse_repo)
df["md_file_name"] = df["workflow_markdown"].apply(parse_workflow) # workflow filed in the metada is just the .md file name. Use this name for another field to map with the actual workflow name
df["md_url"] = df["workflow_markdown"].apply(parse_md_url)
df["yml_url"] = df["workflow_yml"].apply(parse_yml_url)
df["md_sha"] = df["workflow_markdown"].apply(parse_md_SHA)
df["yml_sha"] = df["workflow_yml"].apply(parse_yml_SHA)
df["collected_date"] = df["workflow_markdown"].apply(parse_colected_date)
df["workflow_markdown_without_header"] = df["workflow_markdown"].apply(clean_headers)
df["workflow_yml_without_header"] = df["workflow_yml"].apply(clean_headers)
df["yml_filename"] = df["yml_url"].apply(parse_yml_filename) # use this field to map with workflow runs 
df["gh_aw_id"] = df["workflow_markdown_without_header"].apply(create_gh_aw_id)
df['id'] = df['repo'] + df['gh_aw_id'] # this id should be unique and use to join
# we treat each gh_wh in each repo is a seperated analytic unit even if they have the same .md file. Because we will need to analyze the workflows which are different by repos 

Note: I found 1 .md content file can have multiple .yml content files. Quick check reason is the update of the gh_aw compiler version. So it can be a source of runtime error. But for first analysis iteration. keep it simpple that we create id based on the .md content only

In [145]:
df.head(2)

,workflow_markdown,workflow_yml,repo,md_file_name,md_url,yml_url,md_sha,yml_sha,collected_date,workflow_markdown_without_header,workflow_yml_without_header,yml_filename,gh_aw_id,id
0,# Repo: 0GiS0/github-agentic-workflows-demos\n...,# Repo: 0GiS0/github-agentic-workflows-demos\n...,0GiS0/github-agentic-workflows-demos,label-beautifier,https://github.com/0GiS0/github-agentic-workfl...,https://github.com/0GiS0/github-agentic-workfl...,4141d3afb0f1de3d7f6eee81610a8fc713d8283d,12c91e6631a2f810f7cfa0cd517e151ae2bda8da,2026-03-15T18:34:23.258706,---\n\nname: 💅🏻 Label beautifier\ndescription:...,#\n# ___ _ _ \n# ...,label-beautifier.lock.yml,e5e0f7ae0f7839154d2e4561fcef1a6f802009c79d7085...,0GiS0/github-agentic-workflows-demose5e0f7ae0f...
1,# Repo: 0GiS0/jsconf-2026\n# Workflow: issue-q...,# Repo: 0GiS0/jsconf-2026\n# Workflow: issue-q...,0GiS0/jsconf-2026,issue-quality-enhancer,https://github.com/0GiS0/jsconf-2026/blob/180a...,https://github.com/0GiS0/jsconf-2026/blob/180a...,467d09ae4d582a7353c7d1d7fe8a6684f8386ee0,53a54fb6c92694db5cbaf20d009881f2b1dc6bb5,2026-03-15T18:34:24.714772,---\non:\n issues:\n types: [opened]\n\npe...,#\n# ___ _ _ \n# ...,issue-quality-enhancer.lock.yml,008caacb2a6fc756a9ddbc7ba46ff84ff1e5a6c5414221...,0GiS0/jsconf-2026008caacb2a6fc756a9ddbc7ba46ff...


In [146]:
df["gh_aw_id"].nunique()

3203

In [148]:
df['id'].nunique() # which mean same .md file can sit in different repos

3263

In [149]:
df['collected_date'] = df['collected_date'].astype('datetime64[ns]')

In [150]:
df['ym'] = df['collected_date'].dt.strftime('%Y%m')

In [151]:
df.isnull().sum()

workflow_markdown                   0
workflow_yml                        0
repo                                0
md_file_name                        0
md_url                              0
yml_url                             0
md_sha                              0
yml_sha                             0
collected_date                      0
workflow_markdown_without_header    0
workflow_yml_without_header         0
yml_filename                        0
gh_aw_id                            0
id                                  0
ym                                  0
dtype: int64

In [152]:
df.head(2)

,workflow_markdown,workflow_yml,repo,md_file_name,md_url,yml_url,md_sha,yml_sha,collected_date,workflow_markdown_without_header,workflow_yml_without_header,yml_filename,gh_aw_id,id,ym
0,# Repo: 0GiS0/github-agentic-workflows-demos\n...,# Repo: 0GiS0/github-agentic-workflows-demos\n...,0GiS0/github-agentic-workflows-demos,label-beautifier,https://github.com/0GiS0/github-agentic-workfl...,https://github.com/0GiS0/github-agentic-workfl...,4141d3afb0f1de3d7f6eee81610a8fc713d8283d,12c91e6631a2f810f7cfa0cd517e151ae2bda8da,2026-03-15 18:34:23.258706,---\n\nname: 💅🏻 Label beautifier\ndescription:...,#\n# ___ _ _ \n# ...,label-beautifier.lock.yml,e5e0f7ae0f7839154d2e4561fcef1a6f802009c79d7085...,0GiS0/github-agentic-workflows-demose5e0f7ae0f...,202603
1,# Repo: 0GiS0/jsconf-2026\n# Workflow: issue-q...,# Repo: 0GiS0/jsconf-2026\n# Workflow: issue-q...,0GiS0/jsconf-2026,issue-quality-enhancer,https://github.com/0GiS0/jsconf-2026/blob/180a...,https://github.com/0GiS0/jsconf-2026/blob/180a...,467d09ae4d582a7353c7d1d7fe8a6684f8386ee0,53a54fb6c92694db5cbaf20d009881f2b1dc6bb5,2026-03-15 18:34:24.714772,---\non:\n issues:\n types: [opened]\n\npe...,#\n# ___ _ _ \n# ...,issue-quality-enhancer.lock.yml,008caacb2a6fc756a9ddbc7ba46ff84ff1e5a6c5414221...,0GiS0/jsconf-2026008caacb2a6fc756a9ddbc7ba46ff...,202603


In [153]:
df.shape

(5501, 15)

In [155]:
df['md_url'].nunique() # we cannot use this as a key because its linked to the commit SHA at the collect time

4077

In [156]:
df['ym'].value_counts()

ym
202606    2263
202605     904
202609     827
202608     769
202604     425
202603     313
Name: count, dtype: int64

In [52]:
df.groupby(['md_url'])['ym'].count()

md_url
https://github.com/0GiS0/0gis0/blob/0d9f36cf611a64484a4ede5e8943bdd82dcc7d9e/.github/workflows/smart-labeler.md                                        1
https://github.com/0GiS0/github-agentic-workflows-demos/blob/1a7b83f87c6829313bf579d9651739d6054473db/.github/workflows/continuous-documentation.md    4
https://github.com/0GiS0/github-agentic-workflows-demos/blob/1a7b83f87c6829313bf579d9651739d6054473db/.github/workflows/issue-enhancer.md              1
https://github.com/0GiS0/github-agentic-workflows-demos/blob/1a7b83f87c6829313bf579d9651739d6054473db/.github/workflows/label-beautifier.md            4
https://github.com/0GiS0/github-agentic-workflows-demos/blob/1a7b83f87c6829313bf579d9651739d6054473db/.github/workflows/the-suggester.md               2
                                                                                                                                                      ..
https://github.com/zircote/rlm-rs/blob/72eaedc6304e6d239748cf9e976148227774

because many workflows are duplicated, we use dropduplicate and keep the first collected date only

In [70]:
df.columns

Index(['workflow_markdown', 'workflow_yml', 'repo', 'workflow', 'md_url',
       'yml_url', 'md_SHA', 'yml_SHA', 'collected_date',
       'workflow_markdown_without_header', 'workflow_yml_without_header',
       'ym'],
      dtype='str')

In [158]:
non_dup_df = df.drop(['workflow_markdown', 'workflow_yml', 'collected_date'], axis=1) # we drop this as the header of the files contains collected date which makes them different
non_dup_df = non_dup_df.sort_values(by='ym', ascending=True) # then we keep first in the drop
non_dup_df = non_dup_df.drop_duplicates(subset=['id'], keep='first')

In [159]:
non_dup_df.shape

(3263, 12)

In [160]:
non_dup_df.head(2)

,repo,md_file_name,md_url,yml_url,md_sha,yml_sha,workflow_markdown_without_header,workflow_yml_without_header,yml_filename,gh_aw_id,id,ym
0,0GiS0/github-agentic-workflows-demos,label-beautifier,https://github.com/0GiS0/github-agentic-workfl...,https://github.com/0GiS0/github-agentic-workfl...,4141d3afb0f1de3d7f6eee81610a8fc713d8283d,12c91e6631a2f810f7cfa0cd517e151ae2bda8da,---\n\nname: 💅🏻 Label beautifier\ndescription:...,#\n# ___ _ _ \n# ...,label-beautifier.lock.yml,e5e0f7ae0f7839154d2e4561fcef1a6f802009c79d7085...,0GiS0/github-agentic-workflows-demose5e0f7ae0f...,202603
1,0GiS0/jsconf-2026,issue-quality-enhancer,https://github.com/0GiS0/jsconf-2026/blob/180a...,https://github.com/0GiS0/jsconf-2026/blob/180a...,467d09ae4d582a7353c7d1d7fe8a6684f8386ee0,53a54fb6c92694db5cbaf20d009881f2b1dc6bb5,---\non:\n issues:\n types: [opened]\n\npe...,#\n# ___ _ _ \n# ...,issue-quality-enhancer.lock.yml,008caacb2a6fc756a9ddbc7ba46ff84ff1e5a6c5414221...,0GiS0/jsconf-2026008caacb2a6fc756a9ddbc7ba46ff...,202603


In [161]:
assert non_dup_df.groupby(['id'])['ym'].count().max() == 1 # assert to make sure the df is unique by md_url

In [162]:
non_dup_df.head(2)

,repo,md_file_name,md_url,yml_url,md_sha,yml_sha,workflow_markdown_without_header,workflow_yml_without_header,yml_filename,gh_aw_id,id,ym
0,0GiS0/github-agentic-workflows-demos,label-beautifier,https://github.com/0GiS0/github-agentic-workfl...,https://github.com/0GiS0/github-agentic-workfl...,4141d3afb0f1de3d7f6eee81610a8fc713d8283d,12c91e6631a2f810f7cfa0cd517e151ae2bda8da,---\n\nname: 💅🏻 Label beautifier\ndescription:...,#\n# ___ _ _ \n# ...,label-beautifier.lock.yml,e5e0f7ae0f7839154d2e4561fcef1a6f802009c79d7085...,0GiS0/github-agentic-workflows-demose5e0f7ae0f...,202603
1,0GiS0/jsconf-2026,issue-quality-enhancer,https://github.com/0GiS0/jsconf-2026/blob/180a...,https://github.com/0GiS0/jsconf-2026/blob/180a...,467d09ae4d582a7353c7d1d7fe8a6684f8386ee0,53a54fb6c92694db5cbaf20d009881f2b1dc6bb5,---\non:\n issues:\n types: [opened]\n\npe...,#\n# ___ _ _ \n# ...,issue-quality-enhancer.lock.yml,008caacb2a6fc756a9ddbc7ba46ff84ff1e5a6c5414221...,0GiS0/jsconf-2026008caacb2a6fc756a9ddbc7ba46ff...,202603


In [163]:
non_dup_df.ym.value_counts()

ym
202606    1204
202605     644
202608     428
202609     352
202604     322
202603     313
Name: count, dtype: int64

In [168]:
with pd.option_context('display.max_colwidth', None):
    display(non_dup_df[non_dup_df['md_file_name'] == 'issue-quality-enhancer']['md_url'])

1    https://github.com/0GiS0/jsconf-2026/blob/180aa116c1a817d04f356e592a4f2a028d6d2ea0/.github/workflows/issue-quality-enhancer.md
Name: md_url, dtype: str

In [174]:
assert non_dup_df.shape[0]==(non_dup_df['id'].nunique())

In [176]:
non_dup_df.shape

(3263, 12)

In [183]:
non_dup_df[['repo']].drop_duplicates().reset_index(drop=True).to_csv('unique_repo.csv', index=False)